<a href="https://colab.research.google.com/github/Eight4aWish/Eight4aWish/blob/main/hrcs_coding_gemini_jwb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import google.generativeai as genai
import pandas as pd
from google.colab import userdata

# Function to set the
def set_folder(selected_folder):
  drive.mount('/content/drive', force_remount=True)
  shared_drive_path = "/content/drive/Shareddrives/python_scripts/"
  # create the folder path
  folder_path = shared_drive_path + selected_folder

  # create the folder if it does not exist
  if not os.path.exists(folder_path):
      os.makedirs(folder_path)

  # folder path for ease of use later
  folder_path = folder_path + "/"
  return folder_path



# Mount the shared drive python_scripts, read in the dataset you want
folder_path = set_folder('hrcs_coding_gemini')
dataset = pd.read_excel(folder_path + "coded_hrcs_coding_in.xlsx", nrows = 10)


Mounted at /content/drive


In [ ]:
from time import sleep

# LEAVE
os.environ["GOOGLE_API_KEY"] = userdata.get('gemini_key')  # Set environment variable, secret key needs to be added for your gemini key
genai.configure()  # Configure the library

# CHANGE TO SUIT NEEDS
generation_config = {
  "temperature": 0,   # less creativity with lower value
  "top_p": 0.3,
  "top_k": 10,
  "max_output_tokens": 8192,
  "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
  model_name="gemini-1.5-flash",
  generation_config=generation_config,

)

# LEAVE
chat_session = model.start_chat(

)

# Create an empty list to store the responses
responses = []


# Loop through the dataset and generate responses
#Review the guidance available at https://hrcsonline.net/health-categories/stroke/ on the coding of health research projects. Please assess the project description below and tell me your percentage confidence that the project should be ascribed to the stroke health category in accordance with the guidance {row['Combined_Abstract']}. Base your answer on what is being done in this project. Do not provide any further explanation beyond the percentage.

for _, row in dataset.iterrows():
  prompt = f"""
 You are a health research classification expert tasked with categorising research projects into specific health research categories based on the content of project abstract passages.
    Each project may have one or more research categories. The categories and rules are as follows:

    "Blood":
      "rules": [
        "Is the study primarily focused on diagnosing, treating, or researching blood disorders, blood cell abnormalities, or haemopoietic processes (e.g., anemia, leukemia, clotting disorders)? If yes, use Blood. If the study only mentions blood in the context of broader health evaluations (e.g., blood tests for infectious diseases), do not use Blood.",
        "Is it about the use of direct oral anticoagulants (DOACs) in the context of traumatic intracranial hemorrhage (tICrH) to prevent thrombotic events? If yes, use Blood.",
        "Is it about thrombophlebitis or other conditions primarily involving blood vessels? If yes, do not use Blood (use Cardiovascular).",
        "Does it mention blood in the context of stroke? If yes, do not use Blood (use Stroke).",
        "If the study involves blood tests or blood samples but is primarily focused on diagnosing or evaluating a condition not directly related to blood disorders (e.g., infectious diseases like COVID-19), do not use Blood. Use Blood only if the study is directly related to blood disorders, blood cells, or haemopoietic processes, and not just because blood tests or samples are used. If the study involves blood tests or samples but is not primarily about blood disorders, it should not be categorized under Blood."
      ]
    ,
    "Cancer and neoplasms":
      "rules": [
        "Does the passage discuss smoking or smoking cessation, which is a known risk factor for cancer? If yes, use Cancer and Neoplasms.",
        "Does the passage study cancer-specific risk factors (e.g., smoking, radiation exposure, Barrett's esophagus) or conditions directly linked to cancer development, without focusing on other health conditions? If yes, use Cancer and Neoplasms.",
        "Does the passage exclusively discuss cancer-related biology (e.g., oncogenes, tumor suppressors) or cancer-specific symptoms (e.g., cachexia) without mentioning other health conditions? If yes, use Cancer and Neoplasms.",
        "Does the passage describe a health condition or intervention related to severe obesity that is directly linked to cancer risk or outcomes (e.g., increased risk of certain cancers due to obesity)? If yes, use Cancer and Neoplasms (ignore organ site), else do not use Cancer and Neoplasms."
      ]
    ,
    "Cardiovascular":
      "rules": [
        "Does the passage explicitly focus on conditions primarily affecting the heart or blood vessels (excluding stroke and secondary cardiovascular involvement in other diseases)? If yes, do use Cardiovascular.",
        "Is the project primarily focused on conditions affecting the heart or blood vessels, including but not limited to vascular conditions involving blood clots, such as deep vein thrombosis (DVT)? If yes, do use Cardiovascular.",
        "Does the study involve technologies or methods for measuring health-related quality of life specifically related to cardiovascular conditions or outcomes, such as the EQ-5D-5L? If yes, do use Cardiovascular, else do not use Cardiovascular.",
        "Does the project explicitly and primarily focus on conditions affecting the heart or blood vessels, excluding secondary cardiovascular involvement in other diseases (e.g., complications of cancer or diabetes)? If yes, do use Cardiovascular, else do not use Cardiovascular.",
        "Does the project explicitly and primarily focus on cardiovascular disease or blood vessels, with cardiovascular conditions being the main subject of study (not just complications of another condition or indirectly related outcomes)? If yes, do use Cardiovascular, else do not use Cardiovascular."
      ]
    ,
    "Congenital Disorders":
      "rules": [
        "Does the passage explicitly describe a condition that is present from birth and affects multiple systems, such as a genetic disorder or birth defect? If yes, use the \"Congenital Disorders\" category. If the condition develops later in life or is acquired, do not use this category.",
        "Is the condition present from birth and does it affect multiple systems? If yes, do use Congenital Disorders. If the condition develops during pregnancy, postpartum, or later in life, do not use Congenital Disorders (use the specific system category instead, such as Mental Health for perinatal mental disorders or Obstetrics for pregnancy-related conditions).",
        "Does the project involve a congenital disorder that primarily affects a particular organ system? If yes, use the \"Congenital Disorders\" category."
      ]
    ,
    "Ear":
      "rules": [
        "Does the passage mention British Sign Language (BSL) or communication barriers for Deaf individuals? If yes, use Ear.",
        "Is inner ear function or vestibular disorders the main focus (without brain-level processing)? If yes, use Ear.",
        "Does the study focus on auditory processing or language development in children with auditory neuropathy spectrum disorder (ANSD)? If yes, use Ear.",
        "Does the passage discuss auditory processing or language skills in relation to cleft palate? If yes, use Ear."
      ]
    ,
    "Eye":
      "rules": [
        "Does the passage mention vision, eye structure, or eye movement? If yes, do use Eye",
        "Is the focus on the practical impact of visual impairment (e.g., falls prevention, daily living challenges) rather than brain-level visual processing? If yes, do use Eye.",
        "Does it involve signal transmission from eye to brain? If yes, do use Eye",
        "Does the study primarily focus on eye-related issues (e.g., cataracts, visual symptoms) and their relationship to cognitive decline in elderly patients? If yes, do use Eye."
      ]
    ,
    "Infection":
      "rules": [
        "Does the passage mention a pathogen (virus, bacteria, fungus, parasite) or an infectious disease, including projects focused on diagnostics for these? If yes, use Infection.",
        "Is the project primarily about the infectious agent (e.g., virus, bacteria, fungus, parasite) or the immune response to an infection, with a direct focus on managing or studying these aspects? If yes, do use Infection. If the project focuses on chronic conditions, health systems strengthening, or broader management strategies without emphasizing the infectious agent or immune response, do not use Infection.",
        "Is it about conditions directly caused by infections, such as post-viral fatigue, where the primary focus is on the infection or immune response? If yes, do use Infection. If the focus is on chronic disease management without emphasizing the infectious agent or immune response, do not use Infection.",
        "Is the project about sexual health specifically related to infectious diseases, such as sexually transmitted infections (STIs)? If yes, do use Infection. If the focus is on sexual health without a direct link to infectious diseases, do not use Infection.",
        "Does it focus on immune response to infection or cancer-causing infections? If yes, do use Infection."
      ]
    ,
      "rules": [
        "Is the study primarily focused on understanding or manipulating the immune system or inflammatory responses as a direct mechanism of action or therapeutic target, rather than as a secondary or incidental aspect? If yes, use Inflammatory and Immune System.",
        "Is it about sepsis (any form)? If yes, use Inflammatory and Immune System.",
        "Is the study explicitly focused on normal immune system development or function as its main topic, rather than as a secondary or incidental aspect? If yes, use Inflammatory and Immune System.",
        "Does it describe a strong inflammatory component of another disease as a key aspect of the study? If yes, use Inflammatory and Immune System, else do not use Inflammatory and Immune System."
      ]
    ,
    "Injuries and Accidents":
      "rules": [
        "Does the passage describe injuries or harm resulting from external causes, such as trauma, burns, poisoning, or accidents? If yes, do use Injuries and Accidents.",
        "Is it about wounds or fractures resulting from accidents or trauma (including sports injuries)? If yes, do use Injuries and Accidents.",
        "Does it address prevention of injuries or accidents? If yes, do use Injuries and Accidents.",
        "Is the passage about injuries or harm caused by external events, such as accidents, falls, or sports injuries? If yes, use Injuries and Accidents. If the injuries or harm are due to internal medical conditions, elective surgery, or disease, do not use Injuries and Accidents (use relevant tissue category or Generic Health).",
        "Is it about rehabilitation after injury? If yes, do use Injuries and Accidents (possibly split with the affected system), else do not use Injuries and Accidents."
      ]
    ,
    "Mental Health":
      "rules": [
        "Does the passage mention psychiatric conditions (e.g., depression, schizophrenia), addiction, or substance abuse (including tobacco/nicotine and alcohol)? If yes, do use Mental Health.",
        "Is the study about the decision-making processes of general practitioners (GPs) regarding urgent suspected cancer referrals, including clinical reasoning and patient factors? If yes, do not use Mental Health.",
        "Does the study focus on cognitive processes, such as memory, attention, or decision-making, in individuals with diagnosed mental health disorders? If yes, do use Mental Health.",
        "Is it about psychological aspects of pain perception? If yes, do use Mental Health.",
        "Is it about mental health issues in patients with other conditions (e.g., depression in cancer patients)? If yes, do use Mental Health (split 50/50 with the other condition). Note: General clinical decision-making processes not related to mental health, or studies focusing solely on physiological or biochemical effects of conditions (e.g., traumatic brain injury), do not fall under this category unless they explicitly address mental health issues or psychiatric conditions."
      ]
    ,
    "Metabolic and Endocrine":
      "rules": [
        "Does the passage discuss endocrine glands, hormones, or metabolic disorders (e.g., diabetes)? If yes, do use Metabolic and Endocrine.",
        "Is the project specifically about metabolic or endocrine disorders, including but not limited to elevated blood lipids, cholesterol, HDL, LDL, or triglycerides, or conditions like pre-eclampsia that involve significant metabolic or endocrine changes? If yes, do use Metabolic and Endocrine. If not, do not use Metabolic and Endocrine.",
        "Is the project specifically about metabolic or endocrine disorders related to obesity or dietary metabolism? If yes, do use Metabolic and Endocrine.",
        "Is the project primarily about general nutrition, obesity, or poor diet as a central focus, excluding studies that focus on brain injury, cerebral metabolism, or related treatments unless they directly relate to general nutrition, obesity, or poor diet? If yes, assign 20% to Metabolic and Endocrine. If not, do not use Metabolic and Endocrine.",
        "Does it mention nutrition mainly in relation to metabolic/endocrine function (excluding respiratory function and lung disease)? If yes, do use Metabolic and Endocrine.",
        "Are reproductive hormones central in a metabolic context (not fertility or pregnancy)? If yes, do use Metabolic and Endocrine, else do not use Metabolic and Endocrine."
      ]
    ,
    "Musculoskeletal":
      "rules": [
        "Does the passage primarily discuss bones, joints, or muscles in a way that directly relates to musculoskeletal health? If yes, do use Musculoskeletal, else proceed to the next question.",
        "Is the study primarily focused on disorders that directly affect the muscles, such as muscular dystrophies or myopathies, and not secondary to other conditions like neurological or systemic diseases? If yes, do use Musculoskeletal, else proceed to next question.",
        "Is the study primarily focused on conditions affecting bones, joints, or muscles, such as osteoarthritis, tendonitis, or other musculoskeletal disorders? If yes, do use Musculoskeletal, else proceed to next question.",
        "Is the primary focus on conditions where musculoskeletal pain (e.g., back pain, osteoarthritis, or tendonitis) is the main symptom, and not secondary to other conditions like neuropathic pain or systemic diseases? If yes, do use Musculoskeletal, else proceed to next question.",
        "Is it about sports injuries focused on mechanical/structural aspects of bones or muscles? If yes, do use Musculoskeletal (otherwise see Injuries), else proceed to next question.",
        "Does it focus on connective tissue disorders without a major inflammatory component? If yes, do use Musculoskeletal, else do not use Musculoskeletal."
      ]
    ,
    "Neurological":
      "rules": [
        "Does the passage focus on the biological or physiological aspects of the brain as they relate to the treatment of obsessive-compulsive disorder (OCD) through pharmacological or psychological interventions? If yes, do use Neurological.",
        "Are neurodegenerative conditions (e.g., Alzheimer's, Parkinson's, TSEs, BSE, CJD) mentioned? If yes, do use Neurological.",
        "Is the study focused on the neurological mechanisms (e.g., brain structures, neural pathways) involved in obsessive-compulsive disorder (OCD) and how these mechanisms are targeted by pharmacological or psychological treatments? If yes, do use Neurological.",
        "Is it about sleep disorders or sleep-related movement disorders? If yes, do use Neurological (possibly split with Musculoskeletal).",
        "Does the study explicitly investigate specific neurological mechanisms (e.g., brain structures, neural pathways) underlying obsessive-compulsive disorder (OCD) and how these mechanisms are directly targeted by pharmacological or psychological treatments? If yes, do use Neurological (split 50/50 with Mental Health).",
        "Does the study examine how pharmacological or psychological treatments for obsessive-compulsive disorder (OCD) interact with or affect neurological mechanisms? If yes, do use Neurological (split 50/50 with Mental Health)."
      ]
    ,
    "Oral and Gastrointestinal":
      "rules": [
        "Does the passage mention the digestive tract (mouth, esophagus, stomach, intestines), oral health (including teeth, gums, and any dental issues), or obesity (which significantly affects the gastrointestinal system)? If yes, use Oral and Gastrointestinal.",
        "Does the passage focus on liver health specifically in the context of its role in digestion or gastrointestinal functions, and not in metabolic or endocrine functions? If yes, use Oral and Gastrointestinal.",
        "Does the passage mention oral health (including teeth, gums, and any dental issues) or treatments specifically related to dental care? If yes, use Oral and Gastrointestinal.",
        "Does the passage specifically mention the digestive tract (mouth, esophagus, stomach, intestines), oral health (including teeth, gums, and any dental issues), or liver health (excluding metabolic/endocrine contexts)? If yes, use Oral and Gastrointestinal. If the passage only mentions general alcohol consumption without a specific focus on these areas, do not use Oral and Gastrointestinal."
      ]
    ,
    "Renal and Urogenital":
      "rules": [
        "Does the project involve symptoms or conditions related to the kidneys or urinary tract, such as haematuria, in the context of suspected cancer referrals? If yes, do use Renal and Urogenital, else proceed to next question.",
        "Does the project directly involve the kidneys, urinary tract, or reproductive organs in the context of treatment or study of renal or urogenital diseases? If yes, do use Renal and Urogenital, else proceed to next question.",
        "Is the metabolic/endocrine angle central (e.g., diabetic kidney issues)? If yes, do not use Renal and Urogenital (use Metabolic), else proceed to next question.",
        "Does it discuss STIs specifically? If yes, do not use Renal and Urogenital (use Infection), else do use Renal and Urogenital."
      ]
    ,
    "Reproductive Health and Childbirth":
      "rules": [
        "Does the project primarily focus on any aspect of reproductive health, such as fertility, conception, contraception, pregnancy, childbirth, or postpartum care? If yes, use Reproductive Health and Childbirth. If no, do not use Reproductive Health and Childbirth.",
        "Does the project directly involve childbirth, postpartum care, or lactation/breast function as its primary focus? If yes, do use Reproductive Health and Childbirth. If no, do not use Reproductive Health and Childbirth.",
        "Is the project about sexual health in the context of reproductive health, childbirth, or postpartum care? If yes, do use Reproductive Health and Childbirth (split 50/50 with Infection). If no, do not use Reproductive Health and Childbirth.",
        "Is it about newborn health, including conditions affecting newborns such as Necrotising Enterocolitis (NEC)? If yes, do use Reproductive Health and Childbirth.",
        "Is the project about in-utero exposure effects on later life conditions that are directly related to reproductive health, childbirth, or postpartum care? If yes, do use Reproductive Health and Childbirth. If no, do not use Reproductive Health and Childbirth."
      ]
    ,
    "Respiratory":
      "rules": [
        "Does the passage explicitly mention conditions that directly affect the lungs or airways, such as COPD, pneumonia, bronchitis, or asthma? If yes, do use Respiratory (for asthma, possibly split with Inflammatory). If the passage mentions fever management, pulmonary hypertension, or lung cancer, do not use Respiratory.",
        "Is it about conditions that primarily affect the respiratory system, such as asthma, chronic obstructive pulmonary disease (COPD), or sleep-related breathing disorders (e.g., sleep apnea)? If yes, do use Respiratory. If it is about systemic conditions like sepsis that can affect multiple organs, do not use Respiratory.",
        "Is it a general study about smoking/tobacco without specific disease focus? If yes, assign 25% to Respiratory.",
        "Is it about conditions that primarily affect the respiratory system, such as asthma, chronic obstructive pulmonary disease (COPD), or sleep-related breathing disorders (e.g., sleep apnea)? If yes, do use Respiratory. If it is about systemic conditions like sepsis that can affect multiple organs, do not use Respiratory.",
        "Is it about conditions primarily affecting the respiratory system (e.g., COVID-19, pneumonia, bronchitis)? If yes, do use Respiratory. If it is about pulmonary hypertension, do not use Respiratory (use Cardiovascular).",
        "Does the project involve any conditions or treatments that primarily affect the respiratory system, such as asthma, chronic obstructive pulmonary disease (COPD), or sleep-related breathing disorders (e.g., sleep apnea)? If yes, do use Respiratory. If it is about systemic conditions like obesity that can affect multiple organs, do not use Respiratory."
      ]
    ,
    "Skin":
      "rules": [
        "Does the passage focus on skin conditions (e.g., eczema, psoriasis), rashes, or dermatological health? If yes, do use Skin, else proceed to next question.",
        "Are there any conditions primarily affecting the skin (e.g., eczema, psoriasis) described as the main issue? If yes, do not use Skin (use the appropriate category), else proceed to next question.",
        "Are there any severe skin conditions mentioned (e.g., leprosy, yaws, Buruli ulcer, cutaneous leishmaniasis)? If yes, do use Skin, else proceed to next question.",
        "Are there strong inflammatory aspects directly related to dermatological conditions (e.g., psoriasis)? If yes, do use Skin (possibly split with Inflammatory), else do not use Skin."
      ]
    ,
    "Stroke":
      "rules": [
        "Does the passage explicitly mention the terms \"stroke,\" \"cerebrovascular accident (CVA),\" or \"TIA\"? If yes, assign 100% to Stroke. If no, do not assign Stroke, regardless of any discussion of indirect risk factors such as smoking, cardiovascular disease, or hypertension.",
        "Does the passage explicitly discuss conditions or factors directly linked to stroke risk, such as hypertension, diabetes, or cardiovascular disease, and explicitly connect them to stroke or cerebrovascular health? If yes, assign 50% to Stroke. If the passage mentions these conditions without explicitly linking them to stroke or cerebrovascular health, do not assign Stroke. Proceed to the next question.",
        "Does the passage discuss mental health conditions or treatments that could indirectly impact stroke risk, such as stress or depression? If yes, assign 25% to Stroke, else proceed to the next question.",
        "Does the passage discuss obesity or related health risks that are known to increase the risk of stroke? stroke, such as cardiovascular disease, hypertension, or diabetes? If yes, assign 25% to Stroke, else proceed to the next question.",
        "Does the passage discuss obesity-related health risks, such as cardiovascular disease, hypertension, or diabetes, that are known to increase the risk of stroke? If yes, assign 25% to Stroke, else do not use Stroke."
      ]
    ,
    "Generic Health Relevance":
      "rules": [
        "Does the passage explicitly mention any specific disease category (e.g., cancer, cardiovascular disease) or organ system (e.g., respiratory system, cardiovascular system)? If yes, identify the specific disease category or organ system mentioned and explain why this means the research should not be categorized as Generic Health Relevance.",
        "Does the research involve the development or improvement of methods or tools for measuring health-related quality of life that are applicable across multiple health conditions? If yes, do use Generic Health Relevance.",
        "Is the research applicable to multiple health categories (e.g., Blood, Cancer, Cardiovascular, etc.) without focusing on a specific disease or organ system? If yes, do use Generic Health Relevance.",
        "Is the research focused on developing methods or tools that can be applied to monitor and model health-related data across multiple health conditions, without targeting a specific disease or organ system? If yes, do use Generic Health Relevance.",
        "Is the research applicable to multiple health conditions (e.g., Blood, Cancer, Cardiovascular, etc.) without focusing on a specific disease or organ system? If yes, do use Generic Health Relevance.",
        "Does the research have broad implications for health without focusing on a specific disease, organ system, or health condition? If yes, do use Generic Health Relevance."
      ]
    ,
    "Disputed aetiology and other":
      "rules": [
        "Does the passage explicitly discuss causes of a condition that are either unknown or subject to significant scientific debate, excluding research about treatment effectiveness or known infectious diseases like COVID-19? If yes, use Disputed Aetiology and Other, else proceed to next question.",
        "Does it mention CFS/ME without a clearer fit in another category? If yes, use Disputed Aetiology and Other, else proceed to next question.",
        "Is it about social services research for specific at-risk groups (e.g., substance misuse risk in young people) without other disease focus? If yes, use Disputed Aetiology and Other, else proceed to next question.",
        "Is the research about identifying disputed or unknown causes of a condition, excluding known infectious diseases like COVID-19? If yes, use Disputed Aetiology and Other, else proceed to next question.",
        "Is the research about a known disease or condition? If yes, use Infection, else proceed to next question",
        "Does the research primarily focus on investigating disputed or unknown causes of a condition, excluding known infectious diseases like COVID-19, and does it not fit into any other specific disease category (e.g., Infection, Inflammatory)? If yes, use Disputed Aetiology and Other, else do not use Disputed Aetiology and Other."
      ]
    Carefully analyze the provided project abstract and determine whether they align with the requirements of one or more of the categories based on these criteria:
    Your goal is to ensure accurate and systematic categorisation of research projects for better organisation and retrieval.

Based your answer only on what is being done in this project, not prior or future projects.

Please limit your response to the most relevant categories only.

    Project Abstract:
{row['Combined_Abstract']}.
"""
  response = model.generate_content(prompt)
  responses.append(response.text)
  print(response.text)
  sleep(4) # make sure to sleep, 15 calls per minute for Gemini Flash. 1500 per day.

# Add the responses as a new column to the dataset
dataset['Categories'] = responses

# Print the updated dataset
print(dataset)

#save results
dataset.to_excel(folder_path + "coded_hrcs_coding_out.xlsx", index=False)

KeyboardInterrupt: 